In [1]:
import numpy as np
import tensorflow_datasets as tfds  # pip install tensorflow-datasets
import tensorflow as tf
import matplotlib.pyplot as plt
import keras
from keras import layers
from keras.applications import EfficientNetB0

IMG_SIZE = 224
BATCH_SIZE = 64

In [2]:
dataset_name = "stanford_dogs"
(ds_train, ds_test), ds_info = tfds.load(
    dataset_name, split=["train", "test"], with_info=True, as_supervised=True
)
NUM_CLASSES = ds_info.features["label"].num_classes # 특성의 클래스 개수를 가져옴 (품종 개수)

In [3]:
size = (IMG_SIZE, IMG_SIZE)
ds_train = ds_train.map(lambda image, label: (tf.image.resize(image, size), label))
ds_test = ds_test.map(lambda image, label: (tf.image.resize(image, size), label))

In [4]:
img_augmentation_layers = [
    layers.RandomRotation(factor=0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomFlip(),
    layers.RandomContrast(factor=0.1),
]


def img_augmentation(images):
    for layer in img_augmentation_layers:
        images = layer(images)
    return images

In [5]:
def input_preprocess_train(image, label):
    image = img_augmentation(image)
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label


def input_preprocess_test(image, label):
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label


ds_train = ds_train.map(input_preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
ds_train = ds_train.batch(batch_size=BATCH_SIZE, drop_remainder=True)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.map(input_preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.batch(batch_size=BATCH_SIZE, drop_remainder=True)

In [6]:
def build_model(num_classes):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    model = EfficientNetB0(include_top=False, input_tensor=inputs, weights="imagenet")

    # Freeze the pretrained weights
    model.trainable = False

    # Rebuild top
    x = layers.GlobalAveragePooling2D(name="avg_pool")(model.output)
    x = layers.BatchNormalization()(x)

    top_dropout_rate = 0.2
    x = layers.Dropout(top_dropout_rate, name="top_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="pred")(x)

    # Compile
    model = keras.Model(inputs, outputs, name="EfficientNet")
    optimizer = keras.optimizers.Adam(learning_rate=1e-2)
    model.compile(
        optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"]
    )
    return model

In [7]:
model = build_model(num_classes=NUM_CLASSES)

In [9]:
epochs = 5  # @param {type: "slider", min:8, max:80}
hist = model.fit(ds_train, epochs=epochs, validation_data=ds_test)

Epoch 1/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 174s 935ms/step - accuracy: 0.7098 - loss: 1.0534 - val_accuracy: 0.7926 - val_loss: 0.7885
Epoch 2/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 180s 813ms/step - accuracy: 0.7060 - loss: 1.0563 - val_accuracy: 0.7776 - val_loss: 0.8444
Epoch 3/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 159s 848ms/step - accuracy: 0.7030 - loss: 1.0680 - val_accuracy: 0.7822 - val_loss: 0.8499
Epoch 4/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 150s 803ms/step - accuracy: 0.7047 - loss: 1.0622 - val_accuracy: 0.7812 - val_loss: 0.8731
Epoch 5/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 149s 796ms/step - accuracy: 0.7020 - loss: 1.0725 - val_accuracy: 0.7842 - val_loss: 0.8448


In [10]:
model.save("./saves/stanford_dogs_Transfer_learning.keras")